# Avance 4 — Procesamiento de Datos

## Proyecto Inmobiliario — Región Metropolitana

Este notebook implementa cinco procedimientos de procesamiento
sobre los datos del ecosistema Big Data inmobiliario.

### 1. Limpieza y validación de datos

**Objetivo:** identificar registros duplicados, valores faltantes
y problemas de integridad antes de realizar transformaciones
y análisis.

In [1]:
from pathlib import Path

# Ruta principal del proyecto
RAIZ = Path.cwd().parent

# Carpeta con los datos originales
RUTA_DATOS = RAIZ / "datalake_crudo" / "chilepropiedades"

# Buscar los archivos JSONL
archivos = sorted(RUTA_DATOS.glob("chilepropiedades_*.jsonl"))

print("=== INVENTARIO DE DATOS ===")
print("Archivos encontrados:", len(archivos))

total_registros = 0

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:
        cantidad = sum(1 for linea in f if linea.strip())

    total_registros += cantidad

    print(f"{archivo.name}: {cantidad:,} registros")

print("\nTotal de registros:", f"{total_registros:,}")

=== INVENTARIO DE DATOS ===
Archivos encontrados: 14
chilepropiedades_bodega.jsonl: 141 registros
chilepropiedades_departamento.jsonl: 15,043 registros
chilepropiedades_estudio.jsonl: 20 registros
chilepropiedades_galpon.jsonl: 24 registros
chilepropiedades_hotel.jsonl: 4 registros
chilepropiedades_local-comercial.jsonl: 779 registros
chilepropiedades_loft.jsonl: 5 registros
chilepropiedades_masivo.jsonl: 8,615 registros
chilepropiedades_oficina.jsonl: 1,154 registros
chilepropiedades_parcela.jsonl: 1,065 registros
chilepropiedades_sitio.jsonl: 489 registros
chilepropiedades_terreno-agricola.jsonl: 67 registros
chilepropiedades_terreno-industrial.jsonl: 183 registros
chilepropiedades_terreno.jsonl: 1,014 registros

Total de registros: 28,603


In [2]:
import json
from collections import Counter, defaultdict

conteo_urls = Counter()
archivos_por_url = defaultdict(set)

sin_url = 0
json_invalidos = 0

# Recorrer los 14 archivos
for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for numero_linea, linea in enumerate(f, start=1):

            if not linea.strip():
                continue

            try:
                registro = json.loads(linea)

            except json.JSONDecodeError:
                json_invalidos += 1
                continue

            # Obtener y normalizar la URL
            url = registro.get("url")

            if not isinstance(url, str) or not url.strip():
                sin_url += 1
                continue

            url = url.strip().rstrip("/")

            conteo_urls[url] += 1
            archivos_por_url[url].add(archivo.name)

# Calcular duplicados
urls_duplicadas = {
    url: cantidad
    for url, cantidad in conteo_urls.items()
    if cantidad > 1
}

registros_repetidos = sum(
    cantidad - 1
    for cantidad in urls_duplicadas.values()
)

print("=== CONTROL DE DUPLICADOS ===")
print("Registros originales:", total_registros)
print("URLs distintas:", len(conteo_urls))
print("URLs repetidas:", len(urls_duplicadas))
print("Registros duplicados adicionales:", registros_repetidos)
print("Registros sin URL:", sin_url)
print("JSON inválidos:", json_invalidos)

print("\n=== EJEMPLOS DE DUPLICADOS ===")

for url, cantidad in list(urls_duplicadas.items())[:5]:

    print("\nURL:", url)
    print("Apariciones:", cantidad)
    print("Archivos:", sorted(archivos_por_url[url]))

=== CONTROL DE DUPLICADOS ===
Registros originales: 28603
URLs distintas: 28307
URLs repetidas: 296
Registros duplicados adicionales: 296
Registros sin URL: 0
JSON inválidos: 0

=== EJEMPLOS DE DUPLICADOS ===

URL: https://chilepropiedades.cl/ver-publicacion/venta/estacion-central/estudio/avenida-maria-rozas-velasquez-55/110525656
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/arturo-prat/130710566
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/san-miguel/estudio/avenida-carlos-valdovinos-metro-franklin/118868378
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/garcia-reyes-8/117901234
Apariciones: 2
Archivos: ['chilepropiedades_departa

In [3]:
from collections import defaultdict

# Campos que queremos comparar
campos_comparar = [
    "titulo",
    "precios",
    "caracteristicas",
    "entorno_raw",
    "descripcion_completa"
]

# Recopilar únicamente las publicaciones repetidas
registros_duplicados = defaultdict(list)

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for linea in f:
            if not linea.strip():
                continue

            registro = json.loads(linea)
            url = registro.get("url", "").strip().rstrip("/")

            if url in urls_duplicadas:
                registros_duplicados[url].append({
                    "archivo": archivo.name,
                    "timestamp": registro.get("timestamp"),
                    **{
                        campo: registro.get(campo)
                        for campo in campos_comparar
                    }
                })

# Comparar el contenido de las publicaciones
identicos = 0
diferentes = 0
ejemplos_diferencias = []

for url, registros in registros_duplicados.items():

    campos_diferentes = []

    for campo in campos_comparar:

        valores = {
            json.dumps(
                registro[campo],
                sort_keys=True,
                ensure_ascii=False
            )
            for registro in registros
        }

        if len(valores) > 1:
            campos_diferentes.append(campo)

    if campos_diferentes:
        diferentes += 1
        ejemplos_diferencias.append({
            "url": url,
            "campos": campos_diferentes,
            "archivos": [r["archivo"] for r in registros]
        })
    else:
        identicos += 1

print("=== COMPARACIÓN DE DUPLICADOS ===")
print("URLs con contenido idéntico:", identicos)
print("URLs con diferencias:", diferentes)

print("\n=== EJEMPLOS DE DIFERENCIAS ===")

for ejemplo in ejemplos_diferencias[:5]:
    print("\nURL:", ejemplo["url"])
    print("Campos diferentes:", ejemplo["campos"])
    print("Archivos:", ejemplo["archivos"])

=== COMPARACIÓN DE DUPLICADOS ===
URLs con contenido idéntico: 296
URLs con diferencias: 0

=== EJEMPLOS DE DIFERENCIAS ===


In [4]:
import json
import hashlib
from datetime import datetime, timezone

# Carpeta de salida
RUTA_SILVER = RAIZ / "datalake_procesado" / "silver"
RUTA_SILVER.mkdir(parents=True, exist_ok=True)

archivo_silver = RUTA_SILVER / "chilepropiedades_unicas.jsonl"
archivo_reporte = RUTA_SILVER / "reporte_deduplicacion.json"

urls_vistas = set()

registros_leidos = 0
registros_conservados = 0
duplicados_eliminados = 0

# Crear el nuevo dataset sin modificar los originales
with open(archivo_silver, "w", encoding="utf-8") as salida:

    for archivo in archivos:

        with open(archivo, "r", encoding="utf-8") as entrada:

            for linea in entrada:

                if not linea.strip():
                    continue

                registro = json.loads(linea)
                registros_leidos += 1

                url = registro["url"].strip().rstrip("/")

                # Omitir publicaciones ya encontradas
                if url in urls_vistas:
                    duplicados_eliminados += 1
                    continue

                urls_vistas.add(url)

                # Identificador estable basado en la URL
                id_estable = hashlib.sha256(
                    url.encode("utf-8")
                ).hexdigest()

                registro["url"] = url
                registro["id_publicacion_estable"] = f"CHP-{id_estable}"
                registro["archivo_origen"] = archivo.name

                salida.write(
                    json.dumps(registro, ensure_ascii=False) + "\n"
                )

                registros_conservados += 1

# Reporte de procesamiento
reporte = {
    "procesamiento": "Deduplicación por URL",
    "fecha_procesamiento": datetime.now(
        timezone.utc
    ).isoformat(),
    "archivos_entrada": len(archivos),
    "registros_originales": registros_leidos,
    "registros_conservados": registros_conservados,
    "duplicados_eliminados": duplicados_eliminados,
    "porcentaje_duplicados": round(
        duplicados_eliminados / registros_leidos * 100, 3
    ),
    "criterio": "Conservar la primera aparición por URL normalizada"
}

with open(archivo_reporte, "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=4)

# Validar resultado
assert registros_leidos == (
    registros_conservados + duplicados_eliminados
)

assert registros_conservados == len(urls_vistas)

print("=== RESULTADO DE DEDUPLICACIÓN ===")
print("Registros originales:", registros_leidos)
print("Registros conservados:", registros_conservados)
print("Duplicados eliminados:", duplicados_eliminados)
print("Porcentaje eliminado:", reporte["porcentaje_duplicados"], "%")

print("\nDataset Silver:", archivo_silver)
print("Reporte:", archivo_reporte)

=== RESULTADO DE DEDUPLICACIÓN ===
Registros originales: 28603
Registros conservados: 28307
Duplicados eliminados: 296
Porcentaje eliminado: 1.035 %

Dataset Silver: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_unicas.jsonl
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_deduplicacion.json


### Procesamiento 1 — Limpieza y deduplicación de datos

**Objetivo:** identificar y eliminar registros repetidos entre los 14 archivos JSONL de Chile Propiedades, evitando contabilizar una misma publicación más de una vez.

**Metodología:** se utilizó la URL normalizada como criterio de identificación. Antes de eliminar los duplicados, se compararon los campos `titulo`, `precios`, `caracteristicas`, `entorno_raw` y `descripcion_completa` para comprobar si presentaban diferencias.

**Resultados:**

| Indicador | Resultado |
|---|---:|
| Registros originales | 28.603 |
| URLs distintas | 28.307 |
| Registros duplicados adicionales | 296 |
| Porcentaje eliminado | 1,035 % |
| Registros sin URL | 0 |
| JSON inválidos | 0 |

Se generó el archivo `chilepropiedades_unicas.jsonl` con las publicaciones deduplicadas y el reporte `reporte_deduplicacion.json` con las métricas del procesamiento.

**Conclusión:** la deduplicación permitió reducir la redundancia y obtener una base de publicaciones con URLs distintas. Los archivos originales se conservaron sin modificaciones, manteniendo la trazabilidad entre los datos crudos y los procesados. Esta operación constituye una primera etapa de limpieza, no una validación completa de la calidad de los datos.